# 🎙️ 音声認識実習:Whisperで文字起こしをやってみよう

**授業時間:90分**

この授業では OpenAI の **Whisper** を使って、音声から文字を書き起こすタスクを体験します。

これまで scikit-learn や YOLO を使ってきた人は、実はこの授業でやることの流れは同じです。

| ステップ | scikit-learn (分類) | YOLO (物体検出) | Whisper (音声認識) |
|---|---|---|---|
| 入力 | 特徴量ベクトル | 画像 | 音声波形 |
| 前処理 | 標準化・欠損値処理 | リサイズ・正規化 | メルスペクトログラム変換 |
| モデル | SVM/RandomForest等 | CNN (Darknet等) | Transformer (Encoder-Decoder) |
| 出力 | クラスラベル | バウンディングボックス | テキスト(トークン列) |
| 後処理 | 閾値判定 | NMS(重複除去) | デコード・整形 |

「モデルに何を入れて、何が返ってくるか」という視点で見ると、既に知っているモデルとの共通点が見えてきます。

## 本日の流れ
1. 環境構築(10分)
2. 基本の文字起こし(20分)
3. モデルサイズ比較(15分)
4. 自分の声で試す(15分)
5. 応用:字幕(SRT)生成(15分)
6. まとめ(5分)


## 1. 環境構築(10分)

まずは Whisper をインストールします。Colab には ffmpeg が最初から入っているので、追加設定はほぼ不要です。

GPUを使うと大幅に高速化されます。上部メニューの
**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → T4 GPU**
を選択しておいてください。


In [ ]:
# Whisper をインストール(1〜2分ほどかかります)
!pip install -q openai-whisper

import whisper
import torch

print("PyTorch version:", torch.__version__)
print("GPUが使えるか:", torch.cuda.is_available())


### サンプル音声をダウンロード

著作権フリーのサンプル音声(短い英語スピーチ)を使います。


In [ ]:
# サンプル音声のダウンロード(Whisper公式リポジトリのテスト音声)
!wget -q -O sample.mp3 https://github.com/openai/whisper/raw/main/tests/jfk.flac
!mv sample.mp3 sample.flac 2>/dev/null || true

import IPython.display as ipd
ipd.Audio("sample.flac")


In [ ]:
# 日本語の音声例
#!wget https://www.lab.kochi-tech.ac.jp/~yoshilab/sample_2.flac

In [ ]:
ipd.Audio("sample.flac")

> 🔊 再生ボタンを押して、どんな音声か確認してみましょう。
> 自分の好きな音声ファイルを使いたい場合は、左のフォルダアイコンからアップロードして
> ファイル名を書き換えればOKです。


In [ ]:
!pip install pydub



In [ ]:
import soundfile as sf
filepath = "sample.flac"
data, samplerate = sf.read(filepath)
print(samplerate)

In [ ]:
sr = 44100
filepath = "sample.wav"
sf.write(filepath, data, sr)

In [ ]:
from pydub import AudioSegment

# 音声ファイルの読み込み
audio = AudioSegment.from_wav("sample.wav")


## 2. 基本の文字起こし(20分)

Whisper の使い方はとてもシンプルです。

```python
model = whisper.load_model("base")   # モデルをロード(推論器の準備)
result = model.transcribe("音声ファイル")  # 推論
```

`load_model` は scikit-learn の `model = SomeClassifier()` 、
`transcribe` は `model.predict(X)` に近い感覚で捉えると分かりやすいです。


In [ ]:
# モデルをロード(初回はモデルの重みがダウンロードされます)
model = whisper.load_model("base")

# 文字起こし実行
result = model.transcribe("sample2.flac")

print("認識されたテキスト:")
print(result["text"])


In [ ]:
save_filename = 'speech_out.txt'

with open(save_filename, mode='w') as f:
    f.write(result["text"])

### 結果の中身を詳しく見てみる

`result` には `text` 以外にも、`segments`(区間ごとの情報)や `language`(推定された言語)が入っています。


In [ ]:
print("推定された言語:", result["language"])
print()
print("セグメント数:", len(result["segments"]))
print()

# 最初のセグメントの中身を見てみる
for seg in result["segments"][:3]:
    print(f"[{seg['start']:.2f}s -> {seg['end']:.2f}s] {seg['text']}")


### ✏️ 演習1

1. `result["segments"]` の中には他にどんなキー(情報)が入っているか、`print(result["segments"][0].keys())` で確認してみましょう。
2. `avg_logprob` や `no_speech_prob` はそれぞれ何を表していそうか、YOLOの confidence score と比較しながら考えてみましょう。


In [ ]:
# ここに演習1のコードを書いてみましょう


## 3. モデルサイズ比較(15分)

Whisper には tiny / base / small / medium / large という複数のモデルサイズがあります。
scikit-learn でいう「モデルの複雑さ(木の深さ、ニューラルネットの層数など)」に近い考え方です。

一般的に、**モデルが大きいほど精度は上がるが、推論時間も長くなる**というトレードオフがあります。


In [ ]:
import time

model_names = ["tiny", "base", "small"]
results = {}

for name in model_names:
    print(f"=== {name} モデル ===")
    m = whisper.load_model(name)

    start = time.time()
    r = m.transcribe("sample.flac")
    elapsed = time.time() - start

    results[name] = {"text": r["text"], "time": elapsed}
    print(f"処理時間: {elapsed:.2f}秒")
    print(f"認識結果: {r['text']}")
    print()


In [ ]:
# 結果を表で比較
import pandas as pd

df = pd.DataFrame(results).T
df


### ✏️ 演習2

1. tiny と small で、認識結果のテキストに違いはありましたか?
2. 処理時間はどれくらい違いましたか?数値で比較してみましょう。
3. もしこのモデルを「スマホアプリにリアルタイム組み込みたい」場合と「会議の議事録を高精度に作りたい」場合とで、
   それぞれどのモデルサイズを選びますか?理由も添えて考えてみましょう。


## 4. 自分の声で試す(15分)

Colab上でマイク録音を行うためのコードです(JavaScriptを使ってブラウザのマイクにアクセスします)。
実行するとマイクの使用許可を求められるので「許可」を選んでください。


In [ ]:
# マイク録音用の関数(そのまま実行してOKです)
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(filename="my_voice.wav", seconds=5):
    display(Javascript(RECORD_JS))
    print(f"{seconds}秒間、録音します。話し始めてください…")
    s = output.eval_js(f'record({seconds * 1000})')
    b = b64decode(s.split(',')[1])
    with open(filename, "wb") as f:
        f.write(b)
    print("録音完了:", filename)
    return filename


In [ ]:
# 5秒間、自分の声を録音してみましょう
my_file = record_audio("my_voice.wav", seconds=5)
ipd.Audio(my_file)


In [ ]:
# 録音した自分の声を文字起こし
result_mine = model.transcribe(my_file)
print("あなたの声の認識結果:")
print(result_mine["text"])


> 💡 うまく認識されなかった場合、話す速度・滑舌・雑音などが原因かもしれません。
> 何度か録音し直して、認識結果がどう変わるか試してみましょう。
> 日本語で話した場合、`language` はどう推定されるかも確認してみましょう。


## 5. 応用:字幕(SRT)ファイルを作ってみる(15分)

`segments` には開始時刻・終了時刻が入っているので、動画用の字幕ファイル(SRT形式)を作ることができます。


In [ ]:
def format_timestamp(seconds):
    """秒数を SRT の時刻表記 (00:00:01,234) に変換"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

def make_srt(segments, out_path="output.srt"):
    lines = []
    for i, seg in enumerate(segments, start=1):
        start = format_timestamp(seg["start"])
        end = format_timestamp(seg["end"])
        text = seg["text"].strip()
        lines.append(str(i))
        lines.append(f"{start} --> {end}")
        lines.append(text)
        lines.append("")
    srt_text = "\n".join(lines)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(srt_text)
    return srt_text

srt_text = make_srt(result["segments"])
print(srt_text[:500])


In [ ]:
# 作成したSRTファイルをダウンロード
from google.colab import files
files.download("output.srt")


### 🚀 発展課題(任意・時間が余ったら)

- 日本語音声を英語字幕に翻訳してみる(`model.transcribe(file, task="translate")`)
- 長い音声(数分〜)を使って、実際の講義動画やニュース音声で試してみる
- 文字起こし結果をLLM APIに渡して要約やキーワード抽出をしてみる


## 6. マイクからのリアルタイム(疑似)音声認識

ここまでは「録音 → 完成した音声ファイルを認識」という流れでした。
ここからは、**マイクの音声を数秒ごとに区切りながら、話しているそばから認識結果を表示していく**方式に挑戦します。

USB接続マイクでもPC内蔵マイクでも、ブラウザ(Chrome等)が認識しているデフォルトのマイクがそのまま使われます。
複数のマイクを繋いでいる場合は、ブラウザのアドレスバー左側の🔒アイコン→サイトの設定→マイク、から使用するデバイスを切り替えられます。

### 仕組み
1. ブラウザ側(JavaScript)で `MediaRecorder` を **一定間隔(timeslice)** で回し、音声チャンクを作り続ける
2. チャンクができるたびに `google.colab.kernel.invokeFunction` でPython側の関数を呼び出す
3. Python側でチャンクを受け取り、Whisperで文字起こしして表示する

これは「音声ストリーム → 一定時間窓ごとにモデルへ入力 → 結果を逐次出力」という、
リアルタイム系のML処理でよく使われるスライディングウィンドウの考え方そのものです。


In [ ]:
# Python側:チャンクを受け取って文字起こしする関数を用意
from google.colab import output
from base64 import b64decode
import subprocess, time

streaming_model = whisper.load_model("base")  # ストリーミングには軽量モデルを推奨
chunk_counter = {"n": 0}

def transcribe_chunk(audio_base64):
    chunk_counter["n"] += 1
    n = chunk_counter["n"]

    # ブラウザから届いた音声(webm形式)を保存
    audio_bytes = b64decode(audio_base64.split(',')[1])
    webm_path = f"chunk_{n}.webm"
    wav_path = f"chunk_{n}.wav"
    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    # ffmpegでWhisperが読める形式(wav)に変換
    subprocess.run(
        ["ffmpeg", "-y", "-i", webm_path, "-ar", "16000", wav_path],
        capture_output=True
    )

    result = streaming_model.transcribe(wav_path, language="ja")
    text = result["text"].strip()
    if text:
        print(f"[チャンク{n}] {text}")
    return text

# Python関数をJavaScriptから呼べるように登録
output.register_callback('notebook.transcribe_chunk', transcribe_chunk)
print("コールバック登録完了。次のセルでストリーミングを開始できます。")


In [ ]:
%%javascript
// ブラウザ側:マイクから一定間隔で音声チャンクを取得し、Pythonに送り続ける

window.startStreaming = async function(intervalMs) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);

  recorder.ondataavailable = (e) => {
    const reader = new FileReader();
    reader.onloadend = () => {
      google.colab.kernel.invokeFunction('notebook.transcribe_chunk', [reader.result], {});
    };
    reader.readAsDataURL(e.data);
  };

  recorder.start(intervalMs);  // intervalMs ごとに ondataavailable が発火
  window._recorder = recorder;
  window._stream = stream;
  console.log("ストリーミング開始");
};

window.stopStreaming = function() {
  if (window._recorder) {
    window._recorder.stop();
    window._stream.getTracks().forEach(t => t.stop());
    console.log("ストリーミング停止");
  }
};


### ストリーミング開始・停止

下のセルを実行するとマイクの使用許可が求められます。許可すると **3秒ごと** に区切って認識結果が
上のセルの出力欄(コールバック登録セルの下)に逐次表示されます。

止めたいときは、その次の「停止」セルを実行してください。


In [ ]:
%%javascript
// 3000ms = 3秒ごとにチャンクを送る
startStreaming(3000);


In [ ]:
%%javascript
// ストリーミングを停止する
stopStreaming();


### ✏️ 演習3

1. 実際に日本語で少し長めに話しかけて、3秒ごとの区切りでどこまで正しく認識されるか確認しましょう。
   単語や文の途中でチャンクが切れて、認識結果が不自然になる箇所はありましたか?
2. `startStreaming(3000)` の `3000` を `1500`(1.5秒)や `6000`(6秒)に変えて試してみましょう。
   間隔を短くする/長くすると、認識の反応速度と精度はそれぞれどう変わりますか?
3. (発展)チャンクとチャンクの間に**重なり(オーバーラップ)**を持たせると、境界での切れ目の問題を
   軽減できることが知られています。今回の実装ではオーバーラップさせていませんが、
   どうすれば実現できそうか、アイデアを話し合ってみましょう。

> 💡 補足:これは教育目的の簡易実装です。実務でのリアルタイム音声認識には、
> Whisperのストリーミング対応版(faster-whisperやwhisper.cppのストリーミングモード)や、
> 専用のストリーミングASR API(Google Cloud Speech-to-Text等)がよく使われます。


## まとめ(5分)

今日学んだこと:

- Whisperを使った音声認識の基本フロー(`load_model` → `transcribe`)
- モデルサイズによる精度と速度のトレードオフ
- 音声認識モデルの入出力構造は、これまで学んだ画像分類・物体検出モデルと本質的には同じ「前処理→推論→後処理」のパターンであること
- `segments`の時刻情報を使った字幕生成という実用的な応用
- マイク(USB/内蔵問わず)からのストリーミング風リアルタイム認識と、チャンク分割に伴う精度低下というトレードオフ

### 振り返りの質問
1. 今日一番「なるほど」と思ったポイントは何でしたか?
2. Whisperと、これまで使ったscikit-learnやYOLOとの一番の違いは何だと感じましたか?
3. 音声認識を使ったアプリケーションのアイデアを1つ考えてみましょう。

### 参考リンク
- Whisper 公式リポジトリ: https://github.com/openai/whisper
- Whisper 論文: "Robust Speech Recognition via Large-Scale Weak Supervision"
